In [3]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('articles.csv')

df['text'] = df['title'].fillna('') + ' ' + df['text'].fillna('')

df = df.drop_duplicates(subset=['text'])
docs = df['text'].tolist()

vectorizer = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False)
X = vectorizer.fit_transform(docs)          
vocab = vectorizer.get_feature_names_out()   
print(f"Velikost slovníku: {len(vocab)}")

def load_glove(glove_path):
    glove_dict = {}
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            word = parts[0]
            vector = np.array(parts[1:], dtype=np.float32)
            glove_dict[word] = vector
    return glove_dict

glove_path = 'glove.6B.50d.txt' 
glove_dict = load_glove(glove_path)

d = len(next(iter(glove_dict.values())))

E = np.zeros((len(vocab), d))
missing_count = 0
missing_words = []

for i, word in enumerate(vocab):
    if word in glove_dict:
        E[i] = glove_dict[word]
    else:
        missing_count += 1
        missing_words.append(word)

print(f"Chybí {missing_count} slov z vocab ({missing_count / len(vocab) * 100:.2f} %).")

numerator = X @ E
denominator = np.array(X.sum(axis=1))
denominator[denominator == 0] = 1.0 

doc_vectors = numerator / denominator

query = "coursera vs udacity machine learning"

q_vec = vectorizer.transform([query])
q_numerator = q_vec @ E
q_denominator = np.array(q_vec.sum(axis=1))
q_denominator[q_denominator == 0] = 1.0
q_vector = q_numerator / q_denominator

scores = cosine_similarity(q_vector, doc_vectors)[0]
best_idx = scores.argsort()[::-1][:5]

print("\nTop 5 článků pro dotaz:")
for idx in best_idx:
    print(f"Skóre: {scores[idx]:.4f} | Titulek/Text: {docs[idx][:100]}...")

Velikost slovníku: 41475
Chybí 28354 slov z vocab (68.36 %).

Top 5 článků pro dotaz:
Skóre: 0.5927 | Titulek/Text: 建议的程序员学习LDA算法的步骤 – 蒸汽与魔法 这一阵为了工作上的关系,花了点时间学习了一下LDA算法,说实话,对于我这个学CS而非学数学的人来说,除了集体智慧编程这本书之外基本没怎么看过机器学习的...
Skóre: 0.5671 | Titulek/Text: Multi-Stream RNN, Concat RNN, Internal Conv RNN, Lag 2 RNN in Tensorflow For the last two week I hav...
Skóre: 0.5665 | Titulek/Text: Turn your Raspberry Pi into homemade Google Home – Becoming Human: Artificial Intelligence Magazine ...
Skóre: 0.5577 | Titulek/Text: 1000x Faster Spelling Correction algorithm (2012) – Wolf Garbe – Medium Update1: An improved SymSpel...
Skóre: 0.5556 | Titulek/Text: Machine Learning เรียนอะไร, รู้ไปทําไม – O v e r f i t t e d – Medium Machine Learning จริงๆแล้วมันค...
